# 365 Probabilidades - Dia #122
## Qual a probabilidade de você lembrar o nome de quem acabou de conhecer?

**Tipo:** Comportamental
**Data de publicação:** 2026-10-13
**Ferramenta:** Python
**Decisão analisada:** Quantos nomes ficam depois de uma rodada de apresentações?
**Hashtag:** #365Probabilidades #Dia122

---

### 📖 A História

[Abertura da Ana. O jantar de lançamento do livro, a inserir.]

Fui procurar se alguém já tinha medido quantos nomes ficam na memória depois de uma
apresentação.

---

### 📚 O Conceito: o nome é a parte mais difícil de guardar

Quem acaba de ser apresentado a alguém costuma guardar o rosto e perder o nome. O nome
próprio é uma palavra sem significado ligado à pessoa. Nada no rosto indica que alguém
se chama Marina e não Helena.

Quatro pesquisadores da Universidade da Califórnia em Los Angeles mediram isso de um
jeito próximo de uma apresentação real. Cada participante viu 60 pessoas dizendo "Olá,
meu nome é...", em blocos de 15. Ninguém foi avisado de que haveria teste de memória.
Depois de três minutos de pausa, o participante via o rosto de novo e tinha dez segundos
para escrever o nome.

Em metade das apresentações a pessoa usava um crachá com o nome escrito. Na outra
metade, o crachá estava em branco. O objetivo do estudo era medir o efeito do crachá.
Para a pergunta de hoje interessa primeiro a condição sem crachá, que é a apresentação
comum.

---

### 🧮 O Modelo

Cinco camadas. A fonte dá as médias e os desvios. As contas são nossas.

1. **A chance de um nome.** A proporção de nomes lembrados sem crachá, com intervalo de
   95%, em dois experimentos (vídeo e foto).
2. **As pessoas não são iguais.** O desvio padrão entre participantes, descontado o ruído
   de um teste com 30 nomes, vira uma distribuição Beta da chance de cada pessoa.
3. **Um jantar.** Monte Carlo de uma noite em que se conhecem dez pessoas. Quantos nomes
   ficam, e qual a chance de não ficar nenhum.
4. **O crachá.** O ganho de ver o nome escrito enquanto se ouve, com intervalo de 95%.
5. **Fora do laboratório.** Um estudo feito numa festa real, lido em fonte secundária,
   como referência.

**Fonte central:**
- Murray, C. A., Tarlow, M., Rissman, J. & Shams, L. (2022). "Multisensory encoding of
  names via name tags facilitates remembering". *Applied Cognitive Psychology*, publicação
  online, 1-15. DOI 10.1002/acp.4012. Estudantes de graduação da UCLA, idade média de 19
  anos. 60 pares de rosto e nome em 4 blocos de 15, cada par visto uma vez, 30 com crachá
  e 30 sem. Teste de lembrança com o rosto como pista, 3 minutos depois de cada bloco.
  **Experimento 1 (vídeo):** 38 participantes, 2 excluídos por falha do computador.
  Nomes lembrados sem crachá 0,24 (DP 0,11) e com crachá 0,29 (DP 0,15), t(36) = 3,59,
  d = 0,44. **Experimento 3 (foto com áudio):** 44 participantes, 2 excluídos. Sem
  crachá 0,19 (DP 0,13) e com crachá 0,27 (DP 0,15), t(41) = 4,64, d = 0,57. **Lido no
  texto completo, na cópia publicada pelo laboratório dos autores.**

**Fonte de apoio, lida em fonte secundária:**
- Morris, P. E., Fritz, C. O., Jackson, L., Nichol, E. & Roberts, E. (2005). "Strategies
  for learning proper names: expanding retrieval practice, meaning and imagery". *Applied
  Cognitive Psychology* 19(6), 779-798. DOI 10.1002/acp.1115. Numa festa real, com teste
  de 24 a 72 horas depois, 16% dos nomes lembrados sem técnica e 24% com recuperação
  espaçada. Percentuais lidos em Brédart (2019), *European Psychologist* 24(4).
  **O número de participantes da festa não foi localizado.**

**Nota metodológica.** O fator ×0,80 do projeto não se aplica. Ele vale para proporções
de pesquisa de opinião autorrelatada. Aqui a medida é o desempenho num teste de memória,
corrigido por três avaliadores.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

SEMENTE = 42
SORTEIOS = 200_000

plt.rcParams['font.family'] = 'serif'
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.facecolor'] = 'white'

VERMELHO, DOURADO, VERDE, CINZA = '#c0392b', '#c8a84b', '#1a5f5a', '#6b6a64'
RODAPE = ('Fonte: Murray, Tarlow, Rissman & Shams (2022), Applied Cognitive Psychology · '
          'cálculo próprio | #365Probabilidades')

def mil(n):
    return f"{int(round(n)):,}".replace(",", ".")

def vir(x, casas=1):
    return f"{x:.{casas}f}".replace(".", ",")

print("✅ Bibliotecas carregadas · semente", SEMENTE)

In [ ]:
# --- DADOS DA LITERATURA ---
# Murray, Tarlow, Rissman & Shams (2022), Tabela 1. Proporção de nomes lembrados, média (DP).

NOMES_POR_CONDICAO = 30          # 30 nomes com crachá e 30 sem, por participante
NOMES_POR_BLOCO = 15
PAUSA_MIN = 3

EXPS = {
    'Exp. 1 (vídeo)': dict(n=36, sem=(0.24, 0.11), com=(0.29, 0.15), t=3.59, gl=36, d=0.44),
    'Exp. 3 (foto)':  dict(n=42, sem=(0.19, 0.13), com=(0.27, 0.15), t=4.64, gl=41, d=0.57),
}
# n = inscritos menos excluídos (38 − 2 e 44 − 2). Os graus de liberdade publicados
# (36 e 41) indicariam 37 e 43. A diferença é de uma pessoa e fica declarada.

# Morris et al. (2005), festa real, lido em Brédart (2019). N não localizado.
MORRIS_SEM_TECNICA, MORRIS_RECUPERACAO = 0.16, 0.24

N_TOTAL = sum(e['n'] for e in EXPS.values())

print("=" * 70)
print("  DADOS · MURRAY, TARLOW, RISSMAN & SHAMS (2022)")
print("=" * 70)
print(f"\n  {NOMES_POR_CONDICAO * 2} apresentações por participante, em blocos de {NOMES_POR_BLOCO}")
print(f"  Teste {PAUSA_MIN} minutos depois de cada bloco, com o rosto como pista")
print(f"  Participantes analisados: {N_TOTAL}\n")
for nome, e in EXPS.items():
    print(f"  {nome} · n = {e['n']}")
    print(f"    sem crachá: {vir(e['sem'][0] * 100, 0)}% dos nomes (DP {vir(e['sem'][1] * 100, 0)} pontos)")
    print(f"    com crachá: {vir(e['com'][0] * 100, 0)}% dos nomes (DP {vir(e['com'][1] * 100, 0)} pontos)")
    print(f"    t({e['gl']}) = {vir(e['t'], 2)} · d = {vir(e['d'], 2)}")
print(f"\n  Apoio · Morris et al. (2005), festa real, teste de 24 a 72 horas depois:")
print(f"    sem técnica {vir(MORRIS_SEM_TECNICA * 100, 0)}% · com recuperação espaçada "
      f"{vir(MORRIS_RECUPERACAO * 100, 0)}% · N não localizado")
print("=" * 70)

In [ ]:
# --- O MODELO ---
rng = np.random.default_rng(SEMENTE)

# CAMADA 1 · A chance de um nome (sem crachá)
def ic_media(m, dp, n, nivel=0.95):
    ep = dp / np.sqrt(n)
    tc = stats.t.ppf(0.5 + nivel / 2, n - 1)
    return m - tc * ep, m + tc * ep, ep

print("=" * 70)
print("  CAMADA 1 · A CHANCE DE LEMBRAR UM NOME, SEM CRACHÁ")
print("=" * 70)
ics = {}
for nome, e in EXPS.items():
    lo, hi, ep = ic_media(*e['sem'], e['n'])
    ics[nome] = (lo, hi, ep)
    print(f"  {nome}: {vir(e['sem'][0] * 100)}%  IC 95% [{vir(lo * 100)}; {vir(hi * 100)}]")

P_NOME = sum(e['n'] * e['sem'][0] for e in EXPS.values()) / N_TOTAL
(e1, e3) = EXPS.values()
ep_dif = np.sqrt(sum(v[2] ** 2 for v in ics.values()))
z_dif = (e1['sem'][0] - e3['sem'][0]) / ep_dif
p_dif = 2 * (1 - stats.norm.cdf(abs(z_dif)))
print(f"\n  Média dos dois experimentos, ponderada pelo n: {vir(P_NOME * 100)}%")
print(f"  É cerca de 1 nome em cada {vir(1 / P_NOME)}.")
print(f"  Diferença entre vídeo e foto: {vir((e1['sem'][0] - e3['sem'][0]) * 100, 0)} pontos, "
      f"z = {vir(z_dif, 2)}, p = {vir(p_dif, 2)}")
print("  A diferença não é conclusiva. Os dois resultados ficam como faixa.")

# CAMADA 2 · As pessoas não são iguais
# A variância observada entre participantes soma duas coisas: a diferença real entre
# pessoas e o ruído de um teste com 30 nomes. Desconto o ruído (modelo beta-binomial).
def beta_da_pessoa(m, dp, k=NOMES_POR_CONDICAO):
    var_p = (dp ** 2 - m * (1 - m) / k) / (1 - 1 / k)
    c = m * (1 - m) / var_p - 1
    return m * c, (1 - m) * c, np.sqrt(var_p)

print("\n" + "=" * 70)
print("  CAMADA 2 · AS PESSOAS NÃO SÃO IGUAIS")
print("=" * 70)
betas = {}
for nome, e in EXPS.items():
    a, b, dp_real = beta_da_pessoa(*e['sem'])
    betas[nome] = (a, b)
    dist = stats.beta(a, b)
    q = dist.ppf([0.025, 0.5, 0.975])
    print(f"\n  {nome}")
    print(f"    DP observado {vir(e['sem'][1] * 100, 0)} pontos · DP real entre pessoas "
          f"{vir(dp_real * 100)} pontos")
    print(f"    Beta(α = {vir(a, 2)}, β = {vir(b, 2)})")
    print(f"    95% das pessoas lembram entre {vir(q[0] * 100, 0)}% e {vir(q[2] * 100, 0)}% dos nomes "
          f"(mediana {vir(q[1] * 100, 0)}%)")
    print(f"    lembram menos de 1 em 10: {vir(dist.cdf(0.10) * 100, 0)}% das pessoas")
    print(f"    lembram mais de 1 em 3:   {vir((1 - dist.cdf(1 / 3)) * 100, 0)}% das pessoas")

# Mistura dos dois experimentos, ponderada pelo n
pesos = np.array([e['n'] for e in EXPS.values()]) / N_TOTAL

def sorteia_pessoas(tamanho, condicao='sem'):
    qual = rng.choice(len(EXPS), size=tamanho, p=pesos)
    p = np.empty(tamanho)
    for i, (nome, e) in enumerate(EXPS.items()):
        a, b, _ = beta_da_pessoa(*e[condicao])
        idx = qual == i
        p[idx] = rng.beta(a, b, idx.sum())
    return p

p_pessoas = sorteia_pessoas(SORTEIOS)
print(f"\n  Mistura dos dois experimentos ({mil(SORTEIOS)} pessoas simuladas):")
print(f"    média {vir(p_pessoas.mean() * 100)}% · 95% das pessoas entre "
      f"{vir(np.percentile(p_pessoas, 2.5) * 100, 0)}% e {vir(np.percentile(p_pessoas, 97.5) * 100, 0)}%")
print(f"    lembram menos de 1 em 10: {vir((p_pessoas < 0.10).mean() * 100, 0)}% · "
      f"mais de 1 em 3: {vir((p_pessoas > 1 / 3).mean() * 100, 0)}% · "
      f"mais da metade: {vir((p_pessoas > 0.5).mean() * 100, 1)}%")

# CAMADA 3 · Um jantar
PESSOAS_NO_JANTAR = 10        # premissa nossa
print("\n" + "=" * 70)
print(f"  CAMADA 3 · UM JANTAR (premissa: {PESSOAS_NO_JANTAR} pessoas conhecidas na noite)")
print("=" * 70)
lembrados = rng.binomial(PESSOAS_NO_JANTAR, p_pessoas)
print(f"\n  Nomes lembrados, de {PESSOAS_NO_JANTAR}: média {vir(lembrados.mean())} · "
      f"mediana {int(np.median(lembrados))}")
print(f"  Nenhum nome: {vir((lembrados == 0).mean() * 100)}% das noites")
print(f"  Só um ou dois: {vir(((lembrados >= 1) & (lembrados <= 2)).mean() * 100)}%")
print(f"  Metade ou mais: {vir((lembrados >= PESSOAS_NO_JANTAR / 2).mean() * 100)}%")
print(f"  Todos: {vir((lembrados == PESSOAS_NO_JANTAR).mean() * 100, 3)}%")
print("\n  Conforme o tamanho da roda:")
tabela_jantar = {}
for k in [5, 10, 15, 20]:
    x = rng.binomial(k, p_pessoas)
    tabela_jantar[k] = (x.mean(), (x == 0).mean())
    print(f"    {k:>2} pessoas: {vir(x.mean())} nomes em média · nenhum nome em {vir((x == 0).mean() * 100)}% das noites")
print("  Premissa: a chance por nome não muda com o tamanho da roda. O estudo testou blocos de 15.")

# CAMADA 4 · O crachá
print("\n" + "=" * 70)
print("  CAMADA 4 · O CRACHÁ")
print("=" * 70)
ganhos = {}
for nome, e in EXPS.items():
    dif = e['com'][0] - e['sem'][0]
    ep = dif / e['t']                      # erro padrão da diferença pareada, pelo t publicado
    tc = stats.t.ppf(0.975, e['gl'])
    ganhos[nome] = (dif, dif - tc * ep, dif + tc * ep)
    print(f"  {nome}: +{vir(dif * 100, 0)} pontos  IC 95% [{vir((dif - tc * ep) * 100)}; "
          f"{vir((dif + tc * ep) * 100)}] · {vir(e['com'][0] / e['sem'][0], 2)} vezes a chance sem crachá")
P_NOME_CRACHA = sum(e['n'] * e['com'][0] for e in EXPS.values()) / N_TOTAL
print(f"\n  Média ponderada com crachá: {vir(P_NOME_CRACHA * 100)}% (sem crachá: {vir(P_NOME * 100)}%)")
print(f"  Num jantar com {PESSOAS_NO_JANTAR} pessoas: {vir(PESSOAS_NO_JANTAR * P_NOME)} nomes sem crachá e "
      f"{vir(PESSOAS_NO_JANTAR * P_NOME_CRACHA)} com crachá")
p_cracha = sorteia_pessoas(SORTEIOS, 'com')
lembrados_cracha = rng.binomial(PESSOAS_NO_JANTAR, p_cracha)
print(f"  Nenhum nome, com crachá: {vir((lembrados_cracha == 0).mean() * 100)}% das noites "
      f"(sem crachá: {vir((lembrados == 0).mean() * 100)}%)")
print("  O intervalo usa as médias arredondadas da tabela e o t publicado.")

# CAMADA 5 · Fora do laboratório
print("\n" + "=" * 70)
print("  CAMADA 5 · FORA DO LABORATÓRIO (referência, sem modelo)")
print("=" * 70)
print(f"  Laboratório, 3 minutos depois, rosto como pista: {vir(P_NOME * 100, 0)}%")
print(f"  Festa real, 24 a 72 horas depois (Morris et al., 2005): {vir(MORRIS_SEM_TECNICA * 100, 0)}%")
print(f"  Festa real, com recuperação espaçada: {vir(MORRIS_RECUPERACAO * 100, 0)}%")
print("  Os dois estudos apontam para a mesma ordem de grandeza, de um nome em cada cinco.")
print("  O estudo da festa foi lido em fonte secundária e o N não foi localizado.")
print("=" * 70)

In [ ]:
# --- VISUALIZAÇÃO ---

# GRÁFICO 1 · A chance de um nome
fig1, ax1 = plt.subplots(figsize=(12, 8))
rotulos, medias, erros, cores = [], [], [], []
for nome, e in EXPS.items():
    for cond, cor, txt in [('sem', VERMELHO, 'sem crachá'), ('com', VERDE, 'com crachá')]:
        lo, hi, _ = ic_media(*e[cond], e['n'])
        rotulos.append(f"{nome}\n{txt}")
        medias.append(e[cond][0] * 100)
        erros.append((hi - e[cond][0]) * 100)
        cores.append(cor)
pos = [0, 1, 2.5, 3.5]
ax1.bar(pos, medias, yerr=erros, color=cores, alpha=0.85, width=0.75, capsize=8,
        error_kw=dict(ecolor='#333', lw=1.5))
for x, m, er in zip(pos, medias, erros):
    ax1.text(x, m + er + 1.2, f"{vir(m, 0)}%", ha='center', fontsize=17, fontweight='bold')
ax1.set_xticks(pos)
ax1.set_xticklabels(rotulos, fontsize=11)
ax1.set_ylim(0, 45)
ax1.set_ylabel('Nomes lembrados três minutos depois (%)', fontsize=12)
ax1.set_title('De cada cinco nomes ouvidos numa apresentação, fica um\n'
              'barras de erro: intervalo de 95% da média', fontsize=14, pad=16)
plt.figtext(0.5, 0.01, RODAPE, ha='center', fontsize=9, color='gray')
plt.tight_layout(rect=(0, 0.03, 1, 1))
plt.savefig('dia-122-grafico-01-um-em-cinco.png', dpi=150, bbox_inches='tight')
plt.close()
print("✅ Gráfico 1 salvo!")

# GRÁFICO 2 · As pessoas não são iguais (distribuição Beta)
fig2, ax2 = plt.subplots(figsize=(12, 8))
x = np.linspace(0, 0.7, 800)
for (nome, (a, b)), cor in zip(betas.items(), [VERMELHO, DOURADO]):
    y = stats.beta(a, b).pdf(x)
    ax2.plot(x * 100, y, color=cor, linewidth=3,
             label=f"{nome} · Beta({vir(a, 1)}; {vir(b, 1)})")
    ax2.fill_between(x * 100, y, alpha=0.15, color=cor)
ax2.axvline(P_NOME * 100, color=CINZA, linestyle='--', linewidth=2)
ax2.text(P_NOME * 100 - 3.5, ax2.get_ylim()[1] * 0.93, f"média {vir(P_NOME * 100, 0)}%",
         color=CINZA, fontsize=12, ha='right')
ax2.set_xlabel('Chance de a pessoa lembrar um nome (%)', fontsize=12)
ax2.set_ylabel('Densidade', fontsize=12)
ax2.set_title('A memória para nomes varia muito de pessoa para pessoa\n'
              'distribuição Beta da chance individual, sem crachá', fontsize=14, pad=16)
ax2.legend(frameon=False, fontsize=12)
plt.figtext(0.5, 0.01, RODAPE, ha='center', fontsize=9, color='gray')
plt.tight_layout(rect=(0, 0.03, 1, 1))
plt.savefig('dia-122-grafico-02-as-pessoas.png', dpi=150, bbox_inches='tight')
plt.close()
print("✅ Gráfico 2 salvo!")

# GRÁFICO 3 · Um jantar
fig3, ax3 = plt.subplots(figsize=(12, 8))
ks = np.arange(0, PESSOAS_NO_JANTAR + 1)
freq_sem = np.array([(lembrados == k).mean() for k in ks]) * 100
freq_com = np.array([(lembrados_cracha == k).mean() for k in ks]) * 100
ax3.bar(ks - 0.2, freq_sem, width=0.4, color=VERMELHO, alpha=0.85, label='sem crachá')
ax3.bar(ks + 0.2, freq_com, width=0.4, color=VERDE, alpha=0.85, label='com crachá')
for k in ks:
    if freq_sem[k] >= 1:
        ax3.text(k - 0.2, freq_sem[k] + 0.5, f"{vir(freq_sem[k], 0)}", ha='center', fontsize=10)
    if freq_com[k] >= 1:
        ax3.text(k + 0.2, freq_com[k] + 0.5, f"{vir(freq_com[k], 0)}", ha='center', fontsize=10)
ax3.set_xticks(ks)
ax3.set_xlabel(f'Nomes lembrados, de {PESSOAS_NO_JANTAR} pessoas conhecidas', fontsize=12)
ax3.set_ylabel('Noites simuladas (%)', fontsize=12)
ax3.set_title(f'Um jantar com {PESSOAS_NO_JANTAR} apresentações\n'
              f'{mil(SORTEIOS)} noites simuladas', fontsize=14, pad=16)
ax3.legend(frameon=False, fontsize=12)
plt.figtext(0.5, 0.01, RODAPE, ha='center', fontsize=9, color='gray')
plt.tight_layout(rect=(0, 0.03, 1, 1))
plt.savefig('dia-122-grafico-03-um-jantar.png', dpi=150, bbox_inches='tight')
plt.close()
print("✅ Gráfico 3 salvo!")

# GRÁFICO 4 · O crachá
fig4, ax4 = plt.subplots(figsize=(12, 8))
ys = [1, 0]
for y, (nome, (dif, lo, hi)) in zip(ys, ganhos.items()):
    ax4.plot([lo * 100, hi * 100], [y, y], color=VERDE, linewidth=4)
    ax4.scatter([dif * 100], [y], color=VERDE, s=220, zorder=5)
    ax4.text(dif * 100, y + 0.17, f"+{vir(dif * 100, 0)} pontos  [{vir(lo * 100)}; {vir(hi * 100)}]",
             ha='center', fontsize=13)
ax4.axvline(0, color=CINZA, linewidth=2, linestyle='--')
ax4.text(0.2, 1.62, 'sem efeito', color=CINZA, fontsize=11)
ax4.set_yticks(ys)
ax4.set_yticklabels(list(ganhos.keys()), fontsize=12)
ax4.set_ylim(-0.6, 1.8)
ax4.set_xlim(-2, 14)
ax4.set_xlabel('Ganho em nomes lembrados com o crachá (pontos percentuais)', fontsize=12)
ax4.set_title('Ver o nome escrito enquanto se ouve ajuda um pouco\n'
              'diferença pareada com intervalo de 95%', fontsize=14, pad=16)
plt.figtext(0.5, 0.01, RODAPE, ha='center', fontsize=9, color='gray')
plt.tight_layout(rect=(0, 0.03, 1, 1))
plt.savefig('dia-122-grafico-04-o-cracha.png', dpi=150, bbox_inches='tight')
plt.close()
print("✅ Gráfico 4 salvo!")

### 💡 O Insight

**A chance de lembrar o nome de quem você acabou de conhecer é de 21%.** Um nome em cada
cinco, três minutos depois da apresentação.

O número vem de dois experimentos. Num deles a média foi de 24%, no outro de 19%. A
diferença entre os dois não é conclusiva, e por isso a resposta é uma faixa em torno de
um em cinco.

As pessoas diferem bastante. A maioria fica entre 4% e 43% dos nomes. Uma em cada oito
lembra menos de um nome em dez. Uma em cada oito lembra mais de um em três. Lembrar mais
da metade é raro, e acontece com menos de 1% das pessoas.

Num jantar em que você conhece dez pessoas, a conta dá dois nomes no fim da noite. Em
47% das noites ficam um ou dois. Em 16% não fica nenhum. Lembrar metade ou mais acontece
em 8%.

O crachá ajuda um pouco. Ver o nome escrito enquanto se ouve acrescenta de 5 a 8 pontos.
No jantar de dez pessoas, a média sobe de 2,1 para 2,8 nomes, e a chance de não lembrar
nenhum cai de 16% para 10%.

Fora do laboratório o resultado é parecido. Numa festa real, com teste de um a três dias
depois, um estudo anterior encontrou 16% dos nomes lembrados.

Esquecer o nome de quem acabou de ser apresentado é o resultado mais comum. Acontece
com quatro em cada cinco nomes.

A pergunta que fica:

*De quantas pessoas que você conheceu no último mês você ainda sabe o nome?*

---

### ⚠️ Limitações do Modelo

- **É um experimento de laboratório.** O participante viu 15 apresentações de um ou dois
  segundos em sequência, sem conversar com ninguém. Num jantar há conversa, repetição do
  nome e contexto, e isso pode aumentar a lembrança.
- **O teste foi feito três minutos depois.** O modelo não diz quanto fica no dia seguinte.
- **O teste mostra o rosto e pede o nome.** É a situação de reencontrar a pessoa. Lembrar
  o nome sem ver o rosto é outra tarefa, que o estudo não mediu.
- Os participantes não sabiam que seriam testados. Quem se esforça para guardar um nome
  pode se sair melhor.
- São 78 estudantes de graduação de uma universidade americana, com idade média de 19
  anos, ouvindo nomes comuns nos Estados Unidos.
- **A distribuição entre pessoas é conta nossa.** O artigo publica média e desvio padrão.
  A Beta foi ajustada a esses dois números, descontando o ruído de um teste com 30 nomes.
- **O jantar de dez pessoas é premissa nossa.** O modelo supõe que a chance por nome não
  muda com o número de pessoas apresentadas.
- O intervalo do ganho do crachá usa as médias arredondadas da tabela e o t publicado.
- O número de participantes analisados foi obtido por subtração (inscritos menos
  excluídos). Os graus de liberdade publicados indicam uma pessoa a mais em cada
  experimento.
- O estudo da festa (Morris et al., 2005) foi lido em fonte secundária, e o número de
  participantes não foi localizado. Ele entra só como referência.

*A ciência é honesta sobre o que não sabe. O modelo também.*

---

### 📎 Links
- Substack: [link do post]
- Instagram: [link do post]

---
*365 Probabilidades · Decidindo com dados, um dia de cada vez.*